## (01) Install & Import Dependencies

In [1]:
import os, re, time, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

import gensim
from gensim.models import Word2Vec, FastText, KeyedVectors
import gensim.downloader as gensim_api

from transformers import BertTokenizerFast, BertModel, AutoTokenizer
from torchcrf import CRF
from seqeval.metrics import f1_score, classification_report

warnings.filterwarnings('ignore')

# ─── Reproducibility ─────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')   # ← Apple GPU acceleration
else:
    DEVICE = torch.device('cpu')

print(f'Using device: {DEVICE}')

Using device: mps


## (02) Load BC5CDR Dataset

In [6]:
# ── Set your paths here ──────────────────────────────────────────────────────
TRAIN_FILE = 'train.txt'
VAL_FILE   = 'val.txt'
TEST_FILE  = 'test.txt'

# ─── Parser ───────────────────────────────────────────────────────────────────
def read_conll(filepath):
    """Read CoNLL-style file. Returns list of (tokens, tags) tuples.
    Handles both TAB and SPACE separators, and skips doc-start lines."""
    sentences, tokens, tags = [], [], []
    with open(filepath, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if line.startswith('-DOCSTART-') or line.startswith('#'):
                continue
            if line.strip() == '':
                if tokens:
                    sentences.append((tokens, tags))
                    tokens, tags = [], []
            else:
                parts = line.split('\t') if '\t' in line else line.split()
                if len(parts) >= 2:
                    tokens.append(parts[0])
                    tags.append(parts[-1])   # last column = NER tag
    if tokens:
        sentences.append((tokens, tags))
    return sentences

train_data = read_conll(TRAIN_FILE)
val_data   = read_conll(VAL_FILE)
test_data  = read_conll(TEST_FILE)

print(f'Train sentences : {len(train_data)}')
print(f'Val   sentences : {len(val_data)}')
print(f'Test  sentences : {len(test_data)}')
print('\nSample sentence:')
print(train_data[0])

Train sentences : 4560
Val   sentences : 4581
Test  sentences : 4797

Sample sentence:
(['Selegiline', '-', 'induced', 'postural', 'hypotension', 'in', 'Parkinson', "'", 's', 'disease', ':', 'a', 'longitudinal', 'study', 'on', 'the', 'effects', 'of', 'drug', 'withdrawal', '.'], ['O', 'O', 'O', 'B-Disease', 'I-Disease', 'O', 'B-Disease', 'I-Disease', 'I-Disease', 'I-Disease', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'])



## (03) Build Vocabulary & Tag Mappings

In [7]:
PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'

# ─── Build vocab from TRAIN only (standard NLP practice) ─────────────────────
all_train_tokens = [tok for sent, _ in train_data for tok in sent]
vocab = sorted(set(all_train_tokens))
word2idx = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for w in vocab:
    word2idx[w] = len(word2idx)
idx2word = {v: k for k, v in word2idx.items()}

# ─── Tag mappings ─────────────────────────────────────────────────────────────
all_tags = sorted(set(tag for _, tags in train_data for tag in tags))
tag2idx  = {PAD_TOKEN: 0}
for t in all_tags:
    tag2idx[t] = len(tag2idx)
idx2tag = {v: k for k, v in tag2idx.items()}

VOCAB_SIZE  = len(word2idx)
NUM_CLASSES = len(tag2idx)
print(f'Vocabulary size : {VOCAB_SIZE}')
print(f'Tag set         : {list(tag2idx.keys())}')

Vocabulary size : 9983
Tag set         : ['<PAD>', 'B-Disease', 'I-Disease', 'O']


## (4) Tokenizers

In [8]:
from transformers import AutoTokenizer as HFTokenizer

# ─── Load HuggingFace tokenizer once ─────────────────────────────────────────
HF_MODEL_NAME = 'bert-base-cased'
hf_tokenizer  = HFTokenizer.from_pretrained(HF_MODEL_NAME)

# ─────────────────────────────────────────────────────────────────────────────
def whitespace_tokenize(text):
    """Simple whitespace split."""
    return text.split()

def nltk_tokenize(text):
    """NLTK word tokenizer."""
    return word_tokenize(text)

def hf_tokenize(text):
    """HuggingFace BPE/WordPiece tokenizer — returns subword tokens."""
    return hf_tokenizer.tokenize(text)

TOKENIZERS = {
    'Whitespace' : whitespace_tokenize,
    'NLTK'       : nltk_tokenize,
    'BPE_WP'     : hf_tokenize,
}

# ─── Demo ─────────────────────────────────────────────────────────────────────
sample = 'patients with diabetes mellitus were treated'
for name, fn in TOKENIZERS.items():
    print(f'{name:12s}: {fn(sample)}')

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Whitespace  : ['patients', 'with', 'diabetes', 'mellitus', 'were', 'treated']
NLTK        : ['patients', 'with', 'diabetes', 'mellitus', 'were', 'treated']
BPE_WP      : ['patients', 'with', 'diabetes', 'me', '##lli', '##tus', 'were', 'treated']


## (5) Dataset & Dataloader 

In [9]:
MAX_LEN = 128

class NERDataset(Dataset):
    """
    Dataset for non-BERT embeddings (Word2Vec, GloVe, FastText).
    Tokenizer is applied per-word to get final sub-word indices or
    word indices, then tags are aligned to first sub-token (BIO kept).
    """
    def __init__(self, data, word2idx, tag2idx, tokenizer_fn, max_len=MAX_LEN):
        self.samples = []
        for tokens, tags in data:
            ids, tag_ids = [], []
            for tok, tag in zip(tokens, tags):
                sub = tokenizer_fn(tok)
                if not sub:
                    sub = [tok]
                for i, s in enumerate(sub):
                    ids.append(word2idx.get(s, word2idx.get(tok, word2idx[UNK_TOKEN])))
                    # Only first sub-token keeps real tag; rest get 'O'
                    tag_ids.append(tag2idx.get(tag if i == 0 else 'O', tag2idx['O']))
            ids    = ids[:max_len]
            tag_ids = tag_ids[:max_len]
            self.samples.append((torch.tensor(ids, dtype=torch.long),
                                  torch.tensor(tag_ids, dtype=torch.long)))

    def __len__(self):  return len(self.samples)
    def __getitem__(self, i): return self.samples[i]


def collate_fn(batch):
    """Pad sequences in a batch to equal length."""
    seqs, tags = zip(*batch)
    seqs_padded = pad_sequence(seqs, batch_first=True, padding_value=0)
    tags_padded = pad_sequence(tags, batch_first=True, padding_value=0)
    mask = (seqs_padded != 0).bool()
    return seqs_padded, tags_padded, mask


def make_loaders(train_data, val_data, test_data, word2idx, tag2idx,
                 tokenizer_fn, batch_size=32):
    tr = NERDataset(train_data, word2idx, tag2idx, tokenizer_fn)
    vl = NERDataset(val_data,   word2idx, tag2idx, tokenizer_fn)
    te = NERDataset(test_data,  word2idx, tag2idx, tokenizer_fn)
    return (
        DataLoader(tr, batch_size=batch_size, shuffle=True,  collate_fn=collate_fn),
        DataLoader(vl, batch_size=batch_size, shuffle=False, collate_fn=collate_fn),
        DataLoader(te, batch_size=batch_size, shuffle=False, collate_fn=collate_fn),
    )

print('Dataset utilities defined.')

Dataset utilities defined.


## (6) Embedding Matrices

In [10]:
EMBED_DIM = 100

def build_embedding_matrix(keyed_vectors, word2idx, embed_dim):
    """Build (vocab_size × embed_dim) matrix from gensim KeyedVectors."""
    matrix = np.zeros((len(word2idx), embed_dim))
    hits = 0
    for word, idx in word2idx.items():
        if word in keyed_vectors:
            matrix[idx] = keyed_vectors[word]
            hits += 1
    print(f'  Coverage: {hits}/{len(word2idx)} ({100*hits/len(word2idx):.1f}%)')
    return torch.tensor(matrix, dtype=torch.float)


# ── 6.1 Word2Vec (trained on BC5CDR corpus) ───────────────────────────────────
print('Training Word2Vec on BC5CDR train corpus...')
corpus_sents = [tokens for tokens, _ in train_data]
w2v_model = Word2Vec(sentences=corpus_sents, vector_size=EMBED_DIM,
                     window=5, min_count=1, workers=4, epochs=10, seed=SEED)
w2v_matrix = build_embedding_matrix(w2v_model.wv, word2idx, EMBED_DIM)
print('  Word2Vec done.')


# ── 6.2 GloVe (glove-wiki-gigaword-100 via gensim) ───────────────────────────
print('Loading GloVe (glove-wiki-gigaword-100)...')
glove_kv = gensim_api.load('glove-wiki-gigaword-100')
glove_matrix = build_embedding_matrix(glove_kv, word2idx, EMBED_DIM)
print('  GloVe done.')


# ── 6.3 FastText (trained on BC5CDR corpus) ───────────────────────────────────
print('Training FastText on BC5CDR train corpus...')
ft_model = FastText(sentences=corpus_sents, vector_size=EMBED_DIM,
                    window=5, min_count=1, workers=4, epochs=10, seed=SEED)
ft_matrix = build_embedding_matrix(ft_model.wv, word2idx, EMBED_DIM)
print('  FastText done.')

EMBEDDING_MATRICES = {
    'Word2Vec': w2v_matrix,
    'GloVe'   : glove_matrix,
    'FastText' : ft_matrix,
}
print('\nAll static embeddings ready.')

Training Word2Vec on BC5CDR train corpus...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


  Coverage: 9981/9983 (100.0%)
  Word2Vec done.
Loading GloVe (glove-wiki-gigaword-100)...
[==================================================] 100.0% 128.1/128.1MB downloaded
  Coverage: 6690/9983 (67.0%)
  GloVe done.
Training FastText on BC5CDR train corpus...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


  Coverage: 9983/9983 (100.0%)
  FastText done.

All static embeddings ready.


## (07) Model Architecture — BiLSTM + CRF

In [11]:
class BiLSTM_CRF(nn.Module):
    """
    BiLSTM + CRF NER model for static embeddings (Word2Vec / GloVe / FastText).
    Embedding layer is initialised from a pre-built weight matrix.
    """
    def __init__(self, embed_matrix, hidden_dim, num_tags,
                 dropout=0.3, freeze_embed=False):
        super().__init__()
        vocab_size, embed_dim = embed_matrix.shape

        self.embedding = nn.Embedding.from_pretrained(
            embed_matrix, padding_idx=0, freeze=freeze_embed)
        self.dropout   = nn.Dropout(dropout)
        self.lstm      = nn.LSTM(embed_dim, hidden_dim // 2,
                                  num_layers=2, bidirectional=True,
                                  batch_first=True, dropout=dropout)
        self.fc        = nn.Linear(hidden_dim, num_tags)
        self.crf       = CRF(num_tags, batch_first=True)

    def forward(self, x, tags=None, mask=None):
        emb = self.dropout(self.embedding(x))          # (B, L, E)
        out, _ = self.lstm(emb)                         # (B, L, H)
        emissions = self.fc(self.dropout(out))          # (B, L, T)
        if tags is not None:
            # CRF requires padding_tag=0 to be masked
            loss = -self.crf(emissions, tags, mask=mask, reduction='mean')
            return loss
        return self.crf.decode(emissions, mask=mask)    # list of tag-id seqs

print('BiLSTM_CRF model defined.')

BiLSTM_CRF model defined.


## (08) Training & Evaluation Utilities

In [12]:
def token_accuracy(preds_batch, tags_batch, mask_batch):
    """Per-token accuracy ignoring PAD positions."""
    correct, total = 0, 0
    for preds, tags, mask in zip(preds_batch, tags_batch, mask_batch):
        for p, t, m in zip(preds, tags.tolist(), mask.tolist()):
            if m:
                correct += int(p == t)
                total   += 1
    return correct / total if total > 0 else 0.0


def seqeval_f1(all_preds, all_tags, idx2tag):
    """Compute seqeval F1 from list-of-lists of integer IDs."""
    pred_seqs = [[idx2tag[i] for i in seq] for seq in all_preds]
    true_seqs = [[idx2tag[i] for i in seq] for seq in all_tags]
    return f1_score(true_seqs, pred_seqs)


def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, acc_sum, n_batches = 0, 0, 0
    for x, y, mask in loader:
        x, y, mask = x.to(device), y.to(device), mask.to(device)
        optimizer.zero_grad()
        loss = model(x, tags=y, mask=mask)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item()
        preds = model(x, mask=mask)
        acc_sum += token_accuracy(preds, y.cpu(), mask.cpu())
        n_batches += 1
    return total_loss / n_batches, acc_sum / n_batches


@torch.no_grad()
def evaluate(model, loader, device, idx2tag):
    model.eval()
    acc_sum, n_batches = 0, 0
    all_preds, all_tags = [], []
    for x, y, mask in loader:
        x, y, mask = x.to(device), y.to(device), mask.to(device)
        preds = model(x, mask=mask)           # list of lists
        acc_sum += token_accuracy(preds, y.cpu(), mask.cpu())
        n_batches += 1
        for pred_seq, tag_seq, m in zip(preds, y.cpu().tolist(), mask.cpu().tolist()):
            real_len = sum(m)
            all_preds.append(pred_seq[:real_len])
            all_tags.append(tag_seq[:real_len])
    acc = acc_sum / n_batches
    f1  = seqeval_f1(all_preds, all_tags, idx2tag)
    return acc, f1, all_preds, all_tags


def train_model(model, train_loader, val_loader, test_loader,
                n_epochs=15, lr=1e-3, device=DEVICE):
    """
    Full training loop. Returns history dict with:
      train_acc, val_acc, test_acc, train_loss (per epoch)
    and final val_f1, test_f1.
    """
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=3, factor=0.5)

    history = {'train_acc': [], 'val_acc': [], 'test_acc': [], 'train_loss': []}
    best_val_f1, best_state = 0.0, None

    for epoch in range(1, n_epochs + 1):
        t_loss, t_acc = train_one_epoch(model, train_loader, optimizer, device)
        v_acc,  v_f1, _, _ = evaluate(model, val_loader,  device, idx2tag)
        te_acc, te_f1, _, _ = evaluate(model, test_loader, device, idx2tag)

        scheduler.step(v_f1)
        history['train_loss'].append(t_loss)
        history['train_acc'].append(t_acc)
        history['val_acc'].append(v_acc)
        history['test_acc'].append(te_acc)

        if v_f1 > best_val_f1:
            best_val_f1 = v_f1
            best_state  = {k: v.clone() for k, v in model.state_dict().items()}

        print(f'  Epoch {epoch:02d} | loss {t_loss:.4f} | '
              f'tr_acc {t_acc:.4f} | val_acc {v_acc:.4f} | '
              f'te_acc {te_acc:.4f} | val_F1 {v_f1:.4f}')

    # Restore best weights and get final test metrics
    model.load_state_dict(best_state)
    _, final_val_f1,  _, _  = evaluate(model, val_loader,  device, idx2tag)
    _, final_test_f1, _, _  = evaluate(model, test_loader, device, idx2tag)
    history['val_f1']  = final_val_f1
    history['test_f1'] = final_test_f1
    return history

print('Training utilities defined.')

Training utilities defined.


## (09) BERT Fine-tuning Model & Dataset

In [13]:
class BERTDataset(Dataset):
    """
    Dataset for BERT. Handles sub-word alignment: only first sub-token
    of each word gets the real BIO tag; others get -100 (ignored by loss).
    Supports three tokenization strategies:
      - 'whitespace': join & re-split words, no NLTK
      - 'nltk'      : join & NLTK-tokenize, then BERT-encode
      - 'bpe_wp'    : direct BERT WordPiece (standard)
    """
    def __init__(self, data, tag2idx, hf_tok, mode='bpe_wp', max_len=128):
        self.samples = []
        self.max_len = max_len
        self.hf_tok  = hf_tok
        for tokens, tags in data:
            if mode == 'whitespace':
                words = ' '.join(tokens).split()
                word_tags = self._align_tags(tokens, tags, words)
            elif mode == 'nltk':
                words = word_tokenize(' '.join(tokens))
                word_tags = self._align_tags(tokens, tags, words)
            else:  # bpe_wp — standard
                words, word_tags = tokens, tags

            enc = hf_tok(words, is_split_into_words=True,
                          max_length=max_len, truncation=True, padding='max_length')
            label_ids = []
            prev_word_id = None
            for word_id in enc.word_ids():
                if word_id is None:
                    label_ids.append(-100)
                elif word_id != prev_word_id:
                    label_ids.append(tag2idx.get(word_tags[word_id] if word_id < len(word_tags) else 'O', 0))
                else:
                    label_ids.append(-100)
                prev_word_id = word_id

            self.samples.append({
                'input_ids'      : torch.tensor(enc['input_ids'],      dtype=torch.long),
                'attention_mask' : torch.tensor(enc['attention_mask'],  dtype=torch.long),
                'token_type_ids' : torch.tensor(enc['token_type_ids'],  dtype=torch.long),
                'labels'         : torch.tensor(label_ids,              dtype=torch.long),
            })

    @staticmethod
    def _align_tags(orig_tokens, orig_tags, new_tokens):
        """Greedy character-level re-alignment of tags to new token list."""
        orig_str = ' '.join(orig_tokens)
        new_tags, i = [], 0
        for nt in new_tokens:
            idx = orig_str.find(nt, i)
            char_pos = idx if idx != -1 else i
            # Map char position back to original token index
            cum = 0
            tag = 'O'
            for k, ot in enumerate(orig_tokens):
                if cum <= char_pos < cum + len(ot):
                    tag = orig_tags[k]
                    break
                cum += len(ot) + 1
            new_tags.append(tag)
            i = char_pos + len(nt)
        return new_tags

    def __len__(self):  return len(self.samples)
    def __getitem__(self, i): return self.samples[i]


class BERT_CRF(nn.Module):
    """BERT encoder + linear projection + CRF decoder."""
    def __init__(self, bert_name, num_tags, dropout=0.2):
        super().__init__()
        self.bert    = BertModel.from_pretrained(bert_name)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(self.bert.config.hidden_size, num_tags)
        self.crf     = CRF(num_tags, batch_first=True)

    def forward(self, input_ids, attention_mask, token_type_ids, labels=None):
        out = self.bert(input_ids=input_ids,
                        attention_mask=attention_mask,
                        token_type_ids=token_type_ids)
        emissions = self.fc(self.dropout(out.last_hidden_state))  # (B,L,T)
        mask = attention_mask.bool()
        if labels is not None:
            # Replace -100 with 0 for CRF (mask hides them anyway via attention)
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0
            loss = -self.crf(emissions, safe_labels, mask=mask, reduction='mean')
            return loss
        return self.crf.decode(emissions, mask=mask)


def bert_token_accuracy(preds_batch, labels_batch):
    correct, total = 0, 0
    for preds, labels in zip(preds_batch, labels_batch.tolist()):
        for p, l in zip(preds, labels):
            if l != -100:
                correct += int(p == l)
                total   += 1
    return correct / total if total > 0 else 0.0


def train_bert_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, acc_sum, n = 0, 0, 0
    for batch in loader:
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        ttype = batch['token_type_ids'].to(device)
        labels = batch['labels'].to(device)
        optimizer.zero_grad()
        loss = model(ids, mask, ttype, labels=labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        preds = model(ids, mask, ttype)
        acc_sum += bert_token_accuracy(preds, labels.cpu())
        n += 1
    return total_loss / n, acc_sum / n


@torch.no_grad()
def eval_bert(model, loader, device, idx2tag):
    model.eval()
    acc_sum, n = 0, 0
    all_preds, all_trues = [], []
    for batch in loader:
        ids   = batch['input_ids'].to(device)
        mask  = batch['attention_mask'].to(device)
        ttype = batch['token_type_ids'].to(device)
        labels = batch['labels'].to(device)
        preds  = model(ids, mask, ttype)
        acc_sum += bert_token_accuracy(preds, labels.cpu())
        n += 1
        for pred_seq, label_seq in zip(preds, labels.cpu().tolist()):
            p_tags, t_tags = [], []
            for p, l in zip(pred_seq, label_seq):
                if l != -100:
                    p_tags.append(idx2tag.get(p, 'O'))
                    t_tags.append(idx2tag.get(l, 'O'))
            all_preds.append(p_tags)
            all_trues.append(t_tags)
    f1 = f1_score(all_trues, all_preds)
    return acc_sum / n, f1


def train_bert_model(model, train_loader, val_loader, test_loader,
                     n_epochs=6, lr=2e-5, device=DEVICE):
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.LinearLR(
        optimizer, start_factor=1.0, end_factor=0.1,
        total_iters=n_epochs)

    history = {'train_acc': [], 'val_acc': [], 'test_acc': [], 'train_loss': []}
    best_val_f1, best_state = 0.0, None

    for epoch in range(1, n_epochs + 1):
        t_loss, t_acc = train_bert_epoch(model, train_loader, optimizer, device)
        v_acc,  v_f1  = eval_bert(model, val_loader,  device, idx2tag)
        te_acc, te_f1 = eval_bert(model, test_loader, device, idx2tag)
        scheduler.step()

        history['train_loss'].append(t_loss)
        history['train_acc'].append(t_acc)
        history['val_acc'].append(v_acc)
        history['test_acc'].append(te_acc)

        if v_f1 > best_val_f1:
            best_val_f1 = v_f1
            best_state  = {k: v.clone() for k, v in model.state_dict().items()}

        print(f'  Epoch {epoch:02d} | loss {t_loss:.4f} | '
              f'tr_acc {t_acc:.4f} | val_acc {v_acc:.4f} | '
              f'te_acc {te_acc:.4f} | val_F1 {v_f1:.4f}')

    model.load_state_dict(best_state)
    _, final_val_f1  = eval_bert(model, val_loader,  device, idx2tag)
    _, final_test_f1 = eval_bert(model, test_loader, device, idx2tag)
    history['val_f1']  = final_val_f1
    history['test_f1'] = final_test_f1
    return history

print('BERT model & training utilities defined.')

BERT model & training utilities defined.


## (10) Plotting Utilities

In [14]:
def plot_accuracy_curves(history, title=''):
    """
    Plot train / val / test accuracy on the SAME axes so overfitting is visible.
    If train >> val/test → overfitting; if all three track → healthy.
    """
    epochs = range(1, len(history['train_acc']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # ── Accuracy ────────────────────────────────────────────────────────────
    ax = axes[0]
    ax.plot(epochs, history['train_acc'], 'o-', color='steelblue',  label='Train')
    ax.plot(epochs, history['val_acc'],   's--', color='darkorange', label='Val')
    ax.plot(epochs, history['test_acc'],  '^:', color='forestgreen', label='Test')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Token Accuracy')
    ax.set_title(f'{title}\nAccuracy (overfitting check)')
    ax.legend(); ax.grid(alpha=0.3)
    ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1.0))

    # ── Loss ────────────────────────────────────────────────────────────────
    ax = axes[1]
    ax.plot(epochs, history['train_loss'], 'o-', color='crimson', label='Train Loss')
    ax.set_xlabel('Epoch'); ax.set_ylabel('CRF Loss')
    ax.set_title(f'{title}\nTraining Loss')
    ax.legend(); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
    print(f'  Best Val F1  : {history["val_f1"]:.4f}')
    print(f'  Best Test F1 : {history["test_f1"]:.4f}')

print('Plotting utility defined.')

Plotting utility defined.


In [3]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Must be before ANY other TF import
import tensorflow as tf
tf.compat.v1.enable_eager_execution()

import tensorflow_hub as hub

print(f'TF version: {tf.__version__}')
print(f'Eager mode: {tf.executing_eagerly()}')  # must print True

print('Loading ELMo...')
elmo_tf = hub.load('https://tfhub.dev/google/elmo/3')
ELMO_DIM = 1024
print(f'ELMo loaded.')

TF version: 2.21.0
Eager mode: True
Loading ELMo...
ELMo loaded.


In [4]:
# Quick test — should print a tensor of shape (1, 5, 1024)
test_tokens = ['patients', 'with', 'diabetes', 'mellitus', 'treated']
result = elmo_tf.signatures['tokens'](
    tokens=tf.constant([test_tokens]),
    sequence_len=tf.constant([len(test_tokens)])
)
emb = result['elmo'].numpy()
print(f'Shape: {emb.shape}')   # should be (1, 5, 1024)
print('ELMo is working correctly!')

Shape: (1, 5, 1024)
ELMo is working correctly!
